# 7장 실습 — 환승 설정과 요금 지표

6장의 RAPTOR 는 "언제 도착하는가"만 답했고, 하남시청→미사역이 23분이었습니다.
그 계산에서 정하지 않고 넘어간 값이 있습니다. 환승을 몇 번까지 허용할 것인가, 그 통행에 요금은 얼마인가입니다.
교재 7장에 대응합니다.

이 노트북에서 하는 일은 넷입니다.

1. 수도권 통합환승요금 규칙 세 줄을 코드로 확인합니다 (교재 7.3)
2. 한 통행의 경로와 요금을 끝까지 봅니다
3. 통행 300건의 지표 여섯 개 분포를 봅니다 (교재 7.4, 7.5)
4. 환승 허용 횟수를 늘려 가며 도달 범위가 언제 포화되는지 봅니다 (교재 7.1)

이 장부터는 교재의 정돈본 `smartmob.teaching.raptor` 를 씁니다. 6장에서 여러분이 짠 것과 같은 알고리즘입니다.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import banner, expect, todo
from smartmob.viz import use_korean_font

use_korean_font()

INF = float("inf")

GTFS 는 한 번만 읽고, 자료구조도 한 번만 만듭니다. 이 `data` 와 `origins` 를 노트북 끝까지 씁니다.

In [ ]:
from smartmob.data import load_gtfs
from smartmob.teaching.raptor import TransitData, journey, raptor, summarize

feed = load_gtfs("hanam")
data = TransitData.from_gtfs(feed)
HANAM_CITY_HALL = (37.5393, 127.2148)
origins = data.access_stops(*HANAM_CITY_HALL)       # (정류장 인덱스, 접근 도보 초) 목록

print(f"정류장 {data.n_stops:,}개, 패턴 {len(data.patterns):,}개, 출발 후보 {len(origins)}곳")

## 1. 요금 규칙 (교재 7.3)

교재의 단순화한 요금 모형은 규칙 세 개를 사용합니다.

1. 기본요금은 탄 수단 중 가장 비싼 것 하나만 냅니다 (버스 1,500원, 도시철도 1,550원, GTX 3,200원)
2. 총 이동거리 10km 까지는 기본요금만 냅니다
3. 10km 를 넘으면 5km 마다 100원이 붙습니다 (GTX 를 탔으면 250원)

환승은 요금을 새로 내는 것이 아니라 거리를 이어 붙이는 것입니다.
`calc_fare` 는 구간 목록에서 요금을, `count_transfers` 는 환승 횟수(승차 횟수 − 1)를 돌려줍니다. 도보는 둘 다 세지 않습니다.

In [ ]:
from smartmob.teaching.fare import calc_fare, count_transfers, fare_detail

cases = [
    ("버스 한 번 3km", [{"mode": "BUS", "km": 3.0}]),
    ("버스 한 번 15km", [{"mode": "BUS", "km": 15.0}]),
    ("버스 + 지하철 15km", [{"mode": "BUS", "km": 7.0}, {"mode": "SUBWAY", "km": 8.0}]),
    ("지하철 + GTX 13km", [{"mode": "SUBWAY", "km": 5.0}, {"mode": "GTX", "km": 8.0}]),
    ("걷기만 2km", [{"mode": "WALK", "km": 2.0}]),
]

banner("요금 계산")
for label, legs in cases:
    print(f"{label:22s} {calc_fare(legs):>6,}원  (환승 {count_transfers(legs)}회)")

버스 15km 는 초과 5km 가 한 블록이라 1,600원입니다. 지하철 5km + GTX 8km 는 기본요금 GTX 3,200원에 블록 하나 250원을 더해 3,450원입니다.
`fare_detail` 은 같은 계산을 항목별로 풀어 보여 줍니다. 손으로 검산할 때 씁니다.

In [ ]:
fare_detail([{"mode": "BUS", "km": 7.0}, {"mode": "SUBWAY", "km": 8.0}])

버스 7km + 지하철 8km = 15km. 기본요금은 더 비싼 지하철 1,550원, 초과 5km 는 블록 하나 100원. 합쳐 1,650원입니다.
버스 따로 지하철 따로 내면 3,050원인데 통합요금으로는 1,650원입니다. 이 차이가 환승 할인입니다.

## 2. 한 통행을 끝까지 보기 (교재 7.3)

6장의 하남시청→미사역 통행에 요금을 붙입니다.
`fare_detail` 은 위처럼 적은 사전 목록도 받고, `journey` 가 돌려준 `legs` 도 그대로 받습니다. `legs` 를 주면 구간마다 수단과 거리를 스스로 꺼냅니다.

In [ ]:
def hhmm(seconds):
    return "못 감" if seconds == INF else f"{int(seconds) // 3600:02d}:{int(seconds) % 3600 // 60:02d}"


DEPART = 8 * 3600
result = raptor(data, origins, DEPART)

target = data.nearest_stop(37.5606, 127.1930)            # 미사역 부근
legs = journey(data, result, target)

print(f"하남시청 08:00 출발 → {data.stop_names[target]}")
for leg in legs:
    if leg["kind"] == "transit":
        print(f"  {leg['mode']:7s} {leg['route']:10s} "
              f"{hhmm(leg['board_time'])} → {hhmm(leg['alight_time'])}  {leg['km']:.1f}km")
    else:
        print(f"  WALK    {'':10s} {leg['seconds'] / 60:4.1f}분  {leg['km']:.1f}km")

print()
print(summarize(data, legs, DEPART))
fare_detail(legs)

버스 0.7km + 지하철 2.7km 로 3.4km 통행이라 기본요금 1,550원입니다. 버스와 지하철을 둘 다 탔지만 한 번만 냅니다.

## 3. 지표 여섯 개의 분포 (교재 7.4)

한 통행만 보면 알 수 없습니다. 하남시청에서 갈 수 있는 정류장 300곳을 무작위로 골라 지표를 모읍니다.
`random.Random(7)` 은 교재와 같은 씨앗이라 같은 300곳이 뽑힙니다. 정류장마다 `journey` 로 경로를 되짚고 `summarize` 로 지표를 냅니다.

In [ ]:
import random

import pandas as pd

rng = random.Random(7)
reachable = [i for i, t in enumerate(result.best) if t < INF]
sample = rng.sample(reachable, min(300, len(reachable)))

rows = []
for stop in sample:
    legs = journey(data, result, stop)
    s = summarize(data, legs, DEPART)
    if not s.get("reachable"):
        continue
    rows.append({
        "총_분": s["total_min"],
        "차내_분": s["in_vehicle_min"],
        "도보_분": s["walk_min"],
        "대기_분": s["wait_min"],
        "환승": s["transfers"],
        "요금": calc_fare(legs),
    })

trips = pd.DataFrame(rows)
trips.describe().round(1)

평균 통행시간이 65분인데 표준편차가 66분이고 최댓값이 1,007분입니다. 16시간짜리 통행이 있습니다.
오류가 아닙니다. 하루에 몇 번 안 다니는 노선의 종점이라, 8시에 출발해서 다음 첫차를 기다리는 경우입니다. 중앙값 56분이 훨씬 대표적입니다.
히스토그램으로 여섯 지표를 한 번에 봅니다.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, col in zip(axes.ravel(), trips.columns):
    ax.hist(trips[col], bins=20, color="#4C6EF5", edgecolor="white")
    ax.set_title(col)
plt.tight_layout();

총_분과 대기_분의 오른쪽 꼬리가 길고, 도보_분은 20분 안에 좁게 몰려 있습니다. 목적지가 멀어도 걷는 시간은 비슷하다는 뜻입니다.

## 4. 무엇이 시간을 잡아먹는가 (교재 7.5)

총 통행시간을 차내·도보·대기로 나눠 봅니다. 교재와 같이 3시간을 넘는 극단값은 빼고 셉니다.
버스에 앉아 있는 시간보다 걷고 기다리는 시간이 길면, 개선의 여지는 노선이 아니라 배차 간격과 정류장 접근성에 있습니다.

In [ ]:
normal = trips[trips["총_분"] <= 180]                       # 극단값 제외
share = normal[["차내_분", "도보_분", "대기_분"]].sum()
share = (share / share.sum() * 100).round(1)

banner(f"총 통행시간의 구성 (%)  — 통행 {len(normal)}건")
for name, value in share.items():
    print(f"{name:8s} {value:5.1f}%")

차내가 72%, 도보 14%, 대기 13%입니다. 통행시간의 4분의 1 남짓이 걷고 기다리는 시간입니다.
대기시간은 배차 간격을 줄이면 바로 줄어드는 항목이라, 개선 시나리오를 고를 때 이 분해를 근거로 삼습니다.

## 5. 환승을 몇 번까지 허용할 것인가 (교재 7.1)

RAPTOR 의 `max_rounds` 가 곧 환승 허용 횟수 + 1 입니다. 라운드를 늘려 가며 도달 정류장 수를 셉니다.

In [ ]:
rows = []
for rounds in [1, 2, 3, 4, 5, 6]:
    r = raptor(data, origins, DEPART, max_rounds=rounds)
    reached = sum(1 for t in r.best if t < INF)
    rows.append({
        "라운드": rounds,
        "환승_허용": max(rounds - 1, 0),
        "도달_정류장": reached,
        "도달률": round(reached / data.n_stops, 3),
    })

pd.DataFrame(rows)

환승 없이 갈 수 있는 곳은 1,679개, 한 번 갈아타면 4,152개입니다. 두 번째 환승부터는 4개밖에 늘지 않습니다.

이것은 "갈 수 있느냐"의 이야기입니다. 가장 빠른 경로가 환승 한 번이라는 뜻은 아닙니다.
3절 표본에서 환승 0회는 24건뿐이고 대부분 한 번에서 세 번 갈아탑니다. 환승을 더 허용하면 도달 범위는 그대로여도 도착 시각이 당겨집니다.

## 6. 빈칸

### 6.1 환승을 늘려도 늘지 않는 지점

위 표에서 도달 정류장이 거의 늘지 않기 시작하는 라운드를 적습니다.
실제 경로 안내 서비스가 환승 횟수에 상한을 두는 이유도 두 줄로 적어 봅니다.

In [ ]:
saturation_round = None     # 도달 정류장 증가분이 5개 이하인 첫 라운드

banner("빈칸 6.1")
todo("포화 라운드", saturation_round)

### 6.2 요금이 가장 비싼 통행

`trips` 에서 요금이 가장 비싼 통행의 요금을 찾고, 요금과 총 통행시간의 상관계수를 구합니다.
`trips["요금"].corr(trips["총_분"])` 이 상관계수입니다. 산점도도 그려 요금과 시간이 비례하는지 봅니다.
통합요금은 거리에만 반응하므로, 대기가 긴 통행은 시간이 길어도 요금이 오르지 않습니다.

In [ ]:
max_fare = None         # 가장 비싼 요금 (원)
fare_time_corr = None   # 요금과 총 통행시간의 상관계수

banner("빈칸 6.2")
todo("가장 비싼 요금", max_fare)
todo("요금과 시간의 상관", fare_time_corr, fmt=lambda v: f"{v:.2f}")

### 6.3 출발 시각을 바꾸면

08:00 대신 22:00 에 출발하면 도달 정류장이 얼마나 줄어드는지 구합니다. `raptor` 의 세 번째 인자가 출발 시각(초)입니다.
심야에 갈 수 없는 곳이 어디인지가 이 도시 대중교통의 약점입니다.

In [ ]:
night_reached = None    # 22시 출발 시 도달 정류장 수

banner("빈칸 6.3")
todo("22시 도달 정류장", night_reached)

## 정리

- 통합요금은 거리를 이어 붙여 한 번만 냅니다. 버스 7km + 지하철 8km 가 1,650원입니다
- 하남에서는 환승 한 번이면 도달 정류장이 1,679개에서 4,152개로 늡니다. 두 번째부터는 거의 늘지 않습니다
- 도달 가능한 300곳의 평균 통행시간은 65분인데 중앙값은 56분입니다. 16시간짜리 통행이 평균을 끌어올립니다
- 통행시간을 차내·도보·대기로 쪼개면 무엇을 고쳐야 할지 보입니다. 걷고 기다리는 시간이 4분의 1 남짓입니다
- 8장 실습에서는 택시 쪽으로 돌아가 수요를 직접 만듭니다